<a href="https://colab.research.google.com/github/samipn/AdventurePrototype/blob/master/CRISP_DM_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CRISP-DM: Titanic — End-to-End

Check off each phase as you complete it. Run all cells top-to-bottom in Colab.

In [6]:
#@title Setup
!pip -q install imbalanced-learn fastapi uvicorn joblib plotly
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, joblib, os, json, plotly.express as px
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, classification_report, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
RANDOM_STATE = 42
os.makedirs('data', exist_ok=True)


## Business Understanding
- **Goal**: Predict survival to support prioritization of rescue resources (hypothetical) or demonstrate ML pipeline.
- **Primary KPI**: ROC AUC ≥ 0.86 on validation. Secondary: F1.
- **Constraints**: Model interpretability and reproducibility.

In [7]:
# Acceptance tests (to be validated at the end)
TARGET_KPI = {"auc": 0.86, "f1": 0.78}


### Upload kaggle.json using file browser

Run the following cell to upload your `kaggle.json` file.

In [4]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

Saving kaggle.json to kaggle.json
User uploaded file "kaggle.json" with length 68 bytes


After uploading, proceed to the next steps to configure the Kaggle API and download the dataset.

## Data Understanding
Load data, inspect schema, missingness, target balance.

In [14]:
import os
import shutil

# Get the user's home directory
home_dir = os.path.expanduser("~")

# Create the .kaggle directory if it doesn't exist
kaggle_dir = os.path.join(home_dir, ".kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

# Define the source and destination paths
source_path = "/content/kaggle.json" # Corrected source path
destination_path = os.path.join(kaggle_dir, "kaggle.json")

# Move the kaggle.json file to the .kaggle directory
# Use shutil.move to handle potential overwriting if the file already exists
if os.path.exists(source_path):
    shutil.move(source_path, destination_path)
    print(f"Moved {source_path} to {destination_path}")
else:
    print(f"{source_path} not found. Please ensure it's uploaded to the Colab environment.")

# Set the appropriate permissions for the kaggle.json file
if os.path.exists(destination_path):
    os.chmod(destination_path, 0o600)
    print(f"Set permissions for {destination_path} to 0o600")
else:
    print(f"{destination_path} not found, cannot set permissions.")

/content/kaggle.json not found. Please ensure it's uploaded to the Colab environment.
Set permissions for /root/.kaggle/kaggle.json to 0o600


In [9]:
!kaggle datasets download -d mexwell/heart-disease-dataset -p data --unzip

Dataset URL: https://www.kaggle.com/datasets/mexwell/heart-disease-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
  0% 0.00/399k [00:00<?, ?B/s]
100% 399k/399k [00:00<00:00, 1.16GB/s]


In [12]:
train_path = 'data/heart_statlog_cleveland_hungary_final.csv'
if not os.path.exists(train_path):
    print("Upload Kaggle Titanic train.csv to data/")
df = pd.read_csv(train_path)
df.head()

,age,sex,chest pain type,resting bp s,cholesterol,fasting blood sugar,resting ecg,max heart rate,exercise angina,oldpeak,ST slope,target
0,40,1,2,140,289,0,0,172,0,0.0,1,0
1,49,0,3,160,180,0,0,156,0,1.0,2,1
2,37,1,2,130,283,0,1,98,0,0.0,1,0
3,48,0,4,138,214,0,0,108,1,1.5,2,1
4,54,1,3,150,195,0,0,122,0,0.0,1,0


In [13]:
# Quick profile
display(df.describe(include='all').T)
df.isna().mean().sort_values(ascending=False).head(10)

,count,mean,std,min,25%,50%,75%,max
age,1190.0,53.720168,9.358203,28.0,47.0,54.0,60.00,77.0
sex,1190.0,0.763866,0.424884,0.0,1.0,1.0,1.00,1.0
chest pain type,1190.0,3.232773,0.935480,1.0,3.0,4.0,4.00,4.0
resting bp s,1190.0,132.153782,18.368823,0.0,120.0,130.0,140.00,200.0
cholesterol,1190.0,210.363866,101.420489,0.0,188.0,229.0,269.75,603.0
fasting blood sugar,1190.0,0.213445,0.409912,0.0,0.0,0.0,0.00,1.0
resting ecg,1190.0,0.698319,0.870359,0.0,0.0,0.0,2.00,2.0
max heart rate,1190.0,139.732773,25.517636,60.0,121.0,140.5,160.00,202.0
exercise angina,1190.0,0.387395,0.487360,0.0,0.0,0.0,1.00,1.0
oldpeak,1190.0,0.922773,1.086337,-2.6,0.0,0.6,1.60,6.2


,0
age,0.0
sex,0.0
chest pain type,0.0
resting bp s,0.0
cholesterol,0.0
fasting blood sugar,0.0
resting ecg,0.0
max heart rate,0.0
exercise angina,0.0
oldpeak,0.0


## Data Preparation
Feature engineering: Title extraction, simple imputations, encoding.

In [17]:
feat_df = df.copy() # Use the original dataframe

y = feat_df['target'] # Assuming 'target' is the survival column for this dataset
X = feat_df.drop(columns=['target']) # Drop only the target column

numeric = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical = X.select_dtypes(include=['object','category','bool']).columns.tolist()

num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
pre = ColumnTransformer([('num', num_pipe, numeric), ('cat', cat_pipe, categorical)])

## Modeling
We compare Logistic Regression and Random Forest via stratified CV.

In [18]:
models = {
    "log_reg": LogisticRegression(max_iter=1000, n_jobs=None, random_state=RANDOM_STATE),
    "rf": RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE)
}

from sklearn.model_selection import cross_val_score, StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, clf in models.items():
    pipe = Pipeline([('pre', pre), ('clf', clf)])
    auc = cross_val_score(pipe, X, y, scoring='roc_auc', cv=skf).mean()
    print(name, "cv auc:", round(auc, 4))


log_reg cv auc: 0.9009
rf cv auc: 0.9674


In [19]:
# Fit best model on full train and save
best = Pipeline([('pre', pre), ('clf', RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE))])
best.fit(X, y)
os.makedirs('deployment', exist_ok=True)
joblib.dump(best, 'deployment/model.joblib'); print("Saved to deployment/model.joblib")

Saved to deployment/model.joblib


## Evaluation
Holdout evaluation if you create a split.

In [20]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
best.fit(X_tr, y_tr)
probs = best.predict_proba(X_te)[:,1]
from sklearn.metrics import f1_score
auc = roc_auc_score(y_te, probs)
pred = (probs >= 0.5).astype(int)
f1 = f1_score(y_te, pred)
print("AUC:", auc, "F1:", f1)
print("Meets KPI?", (auc>=TARGET_KPI['auc']) and (f1>=TARGET_KPI['f1']))

AUC: 0.9751275510204082 F1: 0.9285714285714286
Meets KPI? True


## Deployment
Exported model is loaded by FastAPI app under `deployment/api/app.py`. See `deployment/README.md`.